# Azure DevOps & CI/CD Pipelines

Azure DevOps is a suite of developer services that covers the full software delivery lifecycle. It consists of five integrated services:

| Service | Purpose |
|---|---|
| **Azure Repos** | Git repositories (unlimited private repos) |
| **Azure Pipelines** | CI/CD — build, test, and deploy any language to any platform |
| **Azure Boards** | Agile project management — epics, user stories, tasks, sprints |
| **Azure Test Plans** | Manual and exploratory testing with traceability |
| **Azure Artifacts** | Package feeds for NuGet, npm, Maven, Python, and Universal Packages |

Azure Pipelines supports any language (Python, Java, .NET, Node.js, Go, Ruby) and any target (Azure, AWS, GCP, Kubernetes, on-premises). The AWS equivalent is a combination of CodeCommit + CodeBuild + CodePipeline + CodeArtifact.

GitHub Actions is the alternative CI/CD platform for repositories hosted on GitHub — it uses a YAML-based workflow syntax and has deep integration with the GitHub ecosystem. Both Azure Pipelines and GitHub Actions are first-class options for deploying to Azure.

## Azure Pipelines — Core Concepts

### Pipeline Structure

A pipeline is defined in a YAML file (usually `azure-pipelines.yml`) stored in the repository root. The hierarchy is:

```
Pipeline
└── Stage          # logical phase: Build, Test, Deploy-Dev, Deploy-Prod
    └── Job        # runs on a single agent; steps execute sequentially
        └── Step   # individual task or script
```

- **Trigger**: what starts the pipeline — a push to a branch, a pull request, a schedule, or a manual run
- **Stage**: a logical grouping of jobs, e.g. `Build`, `Test`, `Deploy`. Stages run sequentially by default; use `dependsOn` for explicit ordering
- **Job**: runs on a single agent machine. Jobs within a stage can run in parallel
- **Step**: a `task` (a pre-built action) or `script` (an inline shell/PowerShell command)

### Agents

| Agent type | Description | Use when |
|---|---|---|
| **Microsoft-hosted** | Fresh VM per run (ubuntu-latest, windows-latest, macos-latest) | Most builds; zero maintenance |
| **Self-hosted** | Your own VM/container, registered with Azure DevOps | Need specific software, faster builds, access to private network |
| **Scale-set agents** | Azure VMSS pool managed by Azure DevOps | Auto-scale self-hosted agents on demand |

Microsoft-hosted agents include common toolchains pre-installed. Self-hosted agents persist between runs (faster) but require maintenance.

### Variables and Variable Groups

- **Pipeline variables**: defined inline in YAML or in the pipeline UI
- **Variable groups**: reusable collections of variables shared across pipelines, stored in Azure DevOps Library
- **Key Vault–linked variable groups**: a variable group backed by an Azure Key Vault — secrets are fetched at runtime using the service connection's managed identity, never stored in Azure DevOps
- **Secret variables**: marked as secret — their values are masked in logs and never exposed to forked pull requests

## CI Pipeline — Build and Test

A Continuous Integration (CI) pipeline runs on every commit or pull request to ensure the code builds and all tests pass before merge.

### Typical CI stages

1. **Restore** — download dependencies (pip install, npm ci, dotnet restore)
2. **Build** — compile or package the application
3. **Unit Test** — run fast, isolated tests; publish test results and code coverage
4. **Security Scan** — SAST with tools like Checkmarx, Snyk, or the built-in Credential Scanner
5. **Publish Artifact** — upload the build output (Docker image, ZIP, JAR) for use in CD stages

### Pull Request validation

Branch policies in Azure Repos can require:
- A minimum number of reviewer approvals
- A passing build validation pipeline
- Work item linkage
- Comment resolution before merge

### Caching

The `Cache` task stores and restores directories (e.g. `node_modules`, `.pip cache`) between pipeline runs, keyed by a hash of the lock file. This significantly reduces dependency download time on Microsoft-hosted agents.

### Publishing and consuming artifacts

- `PublishPipelineArtifact` — stores build output in Azure DevOps, available to later stages in the same run
- `DownloadPipelineArtifact` — retrieves it in a deployment stage
- `PublishBuildArtifacts` — legacy task; prefer the Pipeline Artifact tasks for better performance

## CD Pipeline — Environments and Deployment Strategies

### Environments

An **Environment** in Azure Pipelines is a named target (e.g. `dev`, `staging`, `production`) that tracks deployments and provides:
- **Deployment history** — every pipeline run that deployed to this environment
- **Approval gates** — require a human to approve before a deployment proceeds
- **Kubernetes resource view** — live pod/workload status when linked to an AKS namespace

### Approvals and checks

Checks on an environment gate the deployment:
- **Manual approval** — one or more named approvers must approve (with a configurable timeout)
- **Azure Monitor alert check** — pipeline waits until no active alerts exist on specified resources
- **REST API check** — calls an external endpoint that must return success
- **Required template** — enforces that the deployment stage uses a specific approved pipeline template
- **Business hours gate** — restrict deployments to specific times

### Deployment strategies

| Strategy | How it works | Use when |
|---|---|---|
| **runOnce** | Deploy to all targets at once | Dev/test environments |
| **Rolling** | Replace targets in batches (e.g. 20% at a time) | Minimise downtime with multiple instances |
| **Canary** | Route a small % of traffic to the new version first | Validate with real traffic before full rollout |
| **Blue-green** | Two identical environments; swap traffic | Zero-downtime with instant rollback |

The `deployment` job type (rather than a regular `job`) integrates with Environments and supports the `strategy` keyword for rolling and canary.

### Service connections

A **service connection** authenticates the pipeline to an external service:
- **Azure Resource Manager** — used to deploy to Azure. Best practice: use **Workload Identity Federation** so no secret is stored in Azure DevOps. The service connection is backed by an app registration in Entra ID with federated credentials that accept tokens issued by Azure DevOps.
- **Docker Registry** — push images to ACR or Docker Hub
- **Kubernetes** — deploy to AKS
- **GitHub, Bitbucket** — check out code from external repos
- **Generic** — any URL with username/password or token

## Deploying to Azure — Common Targets

### App Service

The `AzureWebApp` task (or `AzureRmWebAppDeployment`) deploys a ZIP, JAR, or container image to App Service. Slot deployments let you deploy to a staging slot first, run smoke tests, then swap the staging slot to production — giving zero-downtime deployments with instant rollback by swapping back.

### Azure Container Registry + AKS

A typical container workflow:
1. `Docker@2` task — build and push image to ACR with tag `$(Build.BuildId)`
2. `KubernetesManifest@1` task — apply Kubernetes manifests to AKS, substituting the image tag

Or use Helm: the `HelmDeploy@0` task runs `helm upgrade --install` with values overridden per environment.

### Azure Functions

The `AzureFunctionApp@2` task deploys a ZIP package to an Azure Function App. For Linux Consumption plan, the deployment uses the run-from-package setting.

### Infrastructure as Code

- **ARM/Bicep**: `AzureResourceManagerTemplateDeployment@3` deploys Bicep or ARM templates
- **Terraform**: the `TerraformTaskV4` task (from the Terraform extension) runs `init`, `plan`, and `apply`. Store Terraform state in Azure Blob Storage with state locking via blob leases.

### Azure Artifacts

Internal package feeds let teams publish and consume private packages:
- Upstream sources proxy public registries (PyPI, npmjs, Maven Central) through your feed — all dependencies are cached and auditable
- Package promotion moves a package version through views: `@local` → `@prerelease` → `@release`
- Retention policies automatically delete old package versions to control storage

## GitHub Actions for Azure

GitHub Actions is the CI/CD platform built into GitHub. Workflows are YAML files in `.github/workflows/`. The key concepts map closely to Azure Pipelines:

| GitHub Actions | Azure Pipelines equivalent |
|---|---|
| Workflow | Pipeline |
| Job | Job |
| Step | Step |
| Action (`uses:`) | Task |
| Runner | Agent |
| Environment | Environment |
| Secret | Secret variable |

### Authenticating to Azure from GitHub Actions

Use **OIDC (Workload Identity Federation)** — no long-lived secrets stored in GitHub:
1. Create an app registration in Entra ID with a federated credential trusting GitHub's OIDC issuer
2. Grant the app the required RBAC role on Azure resources
3. In the workflow, use `azure/login@v2` with `client-id`, `tenant-id`, and `subscription-id` — the action exchanges the GitHub-issued OIDC token for an Entra ID access token

### Key Azure actions

| Action | Purpose |
|---|---|
| `azure/login@v2` | Authenticate to Azure via OIDC |
| `azure/arm-deploy@v2` | Deploy ARM/Bicep templates |
| `azure/webapps-deploy@v3` | Deploy to App Service |
| `azure/functions-action@v1` | Deploy to Azure Functions |
| `azure/aks-set-context@v4` | Set kubectl context for AKS |
| `azure/k8s-deploy@v5` | Deploy Kubernetes manifests to AKS |

GitHub Actions environments support required reviewers (approval gates) and deployment protection rules equivalent to Azure Pipelines environment checks.

## Pipeline Security Best Practices

### Secrets management
- Never put secrets in YAML files or source code
- Link variable groups to Azure Key Vault — secrets fetched at runtime
- Use Workload Identity Federation for service connections — no client secrets stored in Azure DevOps
- Restrict secret variable access to specific pipelines and branches

### Supply chain security
- Pin third-party tasks and actions to a specific SHA, not a mutable tag (`@v1`)
- Use Azure Artifacts upstream sources to proxy and audit public package registries
- Run `pip-audit`, `npm audit`, or Snyk in CI to detect vulnerable dependencies
- Enable Credential Scanner (CredScan) in Azure Pipelines to detect secrets accidentally committed to the repo

### Pipeline isolation
- Use separate service connections with the minimum RBAC permissions for each environment (dev/staging/prod)
- Require approval for production deployments via environment checks
- Use **protected resources** — only specific approved pipelines can use production service connections and variable groups
- For self-hosted agents, run each agent pool in an isolated network segment

### Branch protection
- Configure branch policies to block direct pushes to `main`
- Require PR reviews and a passing build validation pipeline before merge
- Use CODEOWNERS (GitHub) or required reviewers (Azure Repos) to enforce ownership

## Code Demo — Azure Pipelines YAML Examples

In [ ]:
# These are YAML pipeline definitions — shown as Python strings for display.
# In practice, save these as azure-pipelines.yml in your repository root.

ci_pipeline = """
# azure-pipelines.yml — CI pipeline for a Python web application
trigger:
  branches:
    include:
      - main
      - feature/*

pr:
  branches:
    include:
      - main

pool:
  vmImage: ubuntu-latest

variables:
  pythonVersion: '3.11'
  containerRegistry: 'myacr.azurecr.io'
  imageRepository: 'myapp'
  imageTag: '$(Build.BuildId)'

stages:
  - stage: Build
    displayName: Build and Test
    jobs:
      - job: BuildTest
        steps:
          # Cache pip dependencies
          - task: Cache@2
            inputs:
              key: 'pip | $(Agent.OS) | requirements.txt'
              restoreKeys: 'pip | $(Agent.OS)'
              path: $(PIP_CACHE_DIR)
            displayName: Cache pip packages

          - task: UsePythonVersion@0
            inputs:
              versionSpec: '$(pythonVersion)'

          - script: pip install -r requirements.txt
            displayName: Install dependencies

          - script: |
              pip install pytest pytest-cov
              pytest tests/ --junitxml=test-results.xml --cov=src --cov-report=xml
            displayName: Run unit tests

          - task: PublishTestResults@2
            inputs:
              testResultsFormat: JUnit
              testResultsFiles: 'test-results.xml'
            condition: always()

          - task: PublishCodeCoverageResults@2
            inputs:
              summaryFileLocation: coverage.xml

          # Build and push Docker image to ACR
          - task: Docker@2
            inputs:
              command: buildAndPush
              repository: '$(imageRepository)'
              dockerfile: Dockerfile
              containerRegistry: myACRServiceConnection
              tags: |
                $(imageTag)
                latest

          - task: PublishPipelineArtifact@1
            inputs:
              targetPath: k8s/
              artifact: manifests
"""

print(ci_pipeline)

In [ ]:
cd_pipeline = """
# CD stages appended to the same pipeline file

  - stage: DeployDev
    displayName: Deploy to Dev
    dependsOn: Build
    condition: succeeded()
    jobs:
      - deployment: DeployAKS
        displayName: Deploy to AKS Dev
        environment: dev  # tracks deployment history; no approval gate on dev
        strategy:
          runOnce:
            deploy:
              steps:
                - download: current
                  artifact: manifests

                - task: KubernetesManifest@1
                  inputs:
                    action: deploy
                    connectionType: azureResourceManager
                    azureSubscriptionConnection: myAzureServiceConnection
                    azureResourceGroup: rg-myapp-dev
                    kubernetesCluster: aks-dev
                    namespace: myapp
                    manifests: |
                      $(Pipeline.Workspace)/manifests/deployment.yaml
                      $(Pipeline.Workspace)/manifests/service.yaml
                    containers: |
                      $(containerRegistry)/$(imageRepository):$(imageTag)

  - stage: DeployProd
    displayName: Deploy to Production
    dependsOn: DeployDev
    condition: and(succeeded(), eq(variables['Build.SourceBranch'], 'refs/heads/main'))
    jobs:
      - deployment: DeployAKSProd
        displayName: Deploy to AKS Prod
        environment: production  # has manual approval gate configured in the portal
        strategy:
          runOnce:
            deploy:
              steps:
                - download: current
                  artifact: manifests

                - task: KubernetesManifest@1
                  inputs:
                    action: deploy
                    connectionType: azureResourceManager
                    azureSubscriptionConnection: myAzureProdServiceConnection
                    azureResourceGroup: rg-myapp-prod
                    kubernetesCluster: aks-prod
                    namespace: myapp
                    manifests: |
                      $(Pipeline.Workspace)/manifests/deployment.yaml
                      $(Pipeline.Workspace)/manifests/service.yaml
                    containers: |
                      $(containerRegistry)/$(imageRepository):$(imageTag)
"""

print(cd_pipeline)

In [ ]:
github_actions_workflow = """
# .github/workflows/deploy.yml — GitHub Actions workflow deploying to App Service
name: Build and Deploy

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

permissions:
  id-token: write   # required for OIDC token exchange
  contents: read

env:
  AZURE_WEBAPP_NAME: myapp-prod
  PYTHON_VERSION: '3.11'

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip

      - run: pip install -r requirements.txt

      - run: pytest tests/ --junitxml=test-results.xml

      - uses: actions/upload-artifact@v4
        with:
          name: app-package
          path: .

  deploy:
    runs-on: ubuntu-latest
    needs: build
    if: github.ref == 'refs/heads/main'
    environment:
      name: production
      url: https://myapp-prod.azurewebsites.net

    steps:
      - uses: actions/download-artifact@v4
        with:
          name: app-package

      # Authenticate via OIDC — no secrets stored in GitHub
      - uses: azure/login@v2
        with:
          client-id: ${{ secrets.AZURE_CLIENT_ID }}
          tenant-id: ${{ secrets.AZURE_TENANT_ID }}
          subscription-id: ${{ secrets.AZURE_SUBSCRIPTION_ID }}

      - uses: azure/webapps-deploy@v3
        with:
          app-name: ${{ env.AZURE_WEBAPP_NAME }}
          package: .
"""

print(github_actions_workflow)

In [ ]:
# Azure DevOps REST API — manage pipelines programmatically
# Install: pip install azure-devops

from azure.devops.connection import Connection
from msrest.authentication import BasicAuthentication
import os

organization_url = "https://dev.azure.com/myorganization"
personal_access_token = os.environ["AZURE_DEVOPS_PAT"]  # never hardcode
project = "MyProject"

credentials = BasicAuthentication("", personal_access_token)
connection = Connection(base_url=organization_url, creds=credentials)

# --- List pipelines ---
pipelines_client = connection.clients.get_pipelines_client()
pipelines = pipelines_client.list_pipelines(project)

print("Pipelines:")
for p in pipelines:
    print(f"  [{p.id}] {p.name}")

# --- Trigger a pipeline run ---
from azure.devops.v7_1.pipelines.models import RunPipelineParameters, RepositoryResourceParameters

pipeline_id = 42  # replace with your pipeline ID
run_params = RunPipelineParameters(
    variables={
        "myVar": {"value": "hello", "isSecret": False}
    }
)
run = pipelines_client.run_pipeline(run_params, project, pipeline_id)
print(f"\nTriggered run ID: {run.id}, state: {run.state}")

# --- List recent builds ---
build_client = connection.clients.get_build_client()
builds = build_client.get_builds(project, top=5)

print("\nRecent builds:")
for b in builds:
    print(f"  [{b.id}] {b.definition.name} — {b.status} / {b.result} — {b.source_branch}")

In [ ]:
# Bicep deployment pipeline task — equivalent YAML snippet

bicep_deploy_task = """
# Deploy infrastructure with Bicep inside an Azure Pipeline stage
- task: AzureResourceManagerTemplateDeployment@3
  displayName: Deploy Bicep template
  inputs:
    deploymentScope: Resource Group
    azureResourceManagerConnection: myAzureServiceConnection
    subscriptionId: $(subscriptionId)
    action: Create Or Update Resource Group
    resourceGroupName: rg-myapp-$(env)
    location: eastus
    templateLocation: Linked artifact
    csmFile: infra/main.bicep
    csmParametersFile: infra/parameters.$(env).json
    deploymentMode: Incremental
    deploymentOutputs: bicepOutputs
"""

terraform_pipeline = """
# Terraform pipeline stages
- stage: TerraformPlan
  jobs:
    - job: Plan
      steps:
        - task: TerraformTaskV4@4
          displayName: Terraform Init
          inputs:
            provider: azurerm
            command: init
            backendServiceArm: myAzureServiceConnection
            backendAzureRmResourceGroupName: rg-tfstate
            backendAzureRmStorageAccountName: mytfstatestorage
            backendAzureRmContainerName: tfstate
            backendAzureRmKey: myapp.tfstate

        - task: TerraformTaskV4@4
          displayName: Terraform Plan
          inputs:
            provider: azurerm
            command: plan
            environmentServiceNameAzureRM: myAzureServiceConnection
            commandOptions: '-out=tfplan'

        - task: PublishPipelineArtifact@1
          inputs:
            targetPath: tfplan
            artifact: tfplan

- stage: TerraformApply
  dependsOn: TerraformPlan
  jobs:
    - deployment: Apply
      environment: production   # requires approval
      strategy:
        runOnce:
          deploy:
            steps:
              - download: current
                artifact: tfplan
              - task: TerraformTaskV4@4
                displayName: Terraform Apply
                inputs:
                  provider: azurerm
                  command: apply
                  environmentServiceNameAzureRM: myAzureServiceConnection
                  commandOptions: '$(Pipeline.Workspace)/tfplan/tfplan'
"""

print("Bicep deployment task:\n", bicep_deploy_task)
print("\nTerraform pipeline stages:\n", terraform_pipeline)

## Summary

| Concept | Key Detail |
|---|---|
| **Azure DevOps services** | Repos, Pipelines, Boards, Test Plans, Artifacts |
| **Pipeline hierarchy** | Pipeline → Stage → Job → Step |
| **Agent types** | Microsoft-hosted (fresh VM), Self-hosted, Scale-set |
| **CI best practices** | Cache dependencies, publish test results and coverage, scan for secrets |
| **Environments** | Track deployments, enforce approvals and checks per environment |
| **Deployment strategies** | runOnce, Rolling, Canary, Blue-green |
| **Service connections** | Authenticate pipeline to Azure — use Workload Identity Federation (no secrets) |
| **Key Vault variables** | Link variable group to Key Vault — secrets fetched at runtime |
| **GitHub Actions** | OIDC login via `azure/login@v2` — no long-lived secrets |
| **IaC deployment** | Bicep via `AzureResourceManagerTemplateDeployment@3`; Terraform via `TerraformTaskV4` |
| **Azure Artifacts** | Private feeds with upstream proxy for auditable supply chain |
| **Branch policies** | Require PR review + passing build before merge to main |